the keywords and main movie dataset are two different csv files but i will join them based on id to make things easier

In [303]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
import ast
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Set the path to the file you'd like to load
file_path = "movies_metadata.csv"

keywords_path = "keywords.csv"

df2 = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "rounakbanik/the-movies-dataset",
  keywords_path,
)


# Load the latest version
df1 = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "rounakbanik/the-movies-dataset",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)
df1['id'] = df1['id'].astype(str)
df2['id'] = df2['id'].astype(str)
df = pd.merge(df1, df2, on='id', how='inner')


print("First 5 records:", df.head())

First 5 records:    adult                              belongs_to_collection    budget  \
0  False  {'id': 10194, 'name': 'Toy Story Collection', ...  30000000   
1  False                                                NaN  65000000   
2  False  {'id': 119050, 'name': 'Grumpy Old Men Collect...         0   
3  False                                                NaN  16000000   
4  False  {'id': 96871, 'name': 'Father of the Bride Col...         0   

                                              genres  \
0  [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   
1  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   
2  [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...   
3  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...   
4                     [{'id': 35, 'name': 'Comedy'}]   

                               homepage     id    imdb_id original_language  \
0  http://toystory.disney.com/toy-story    862  tt0114709                en   
1                                   NaN   8844  t

/Users/sanakulkarni/Datasets/Movie-Recommendation/venv/lib/python3.11/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (0: popularity) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [304]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 46482 entries, 0 to 46481
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  46482 non-null  str    
 1   belongs_to_collection  4557 non-null   str    
 2   budget                 46482 non-null  str    
 3   genres                 46482 non-null  str    
 4   homepage               7986 non-null   str    
 5   id                     46482 non-null  str    
 6   imdb_id                46465 non-null  str    
 7   original_language      46471 non-null  str    
 8   original_title         46482 non-null  str    
 9   overview               45487 non-null  str    
 10  popularity             46478 non-null  object 
 11  poster_path            46083 non-null  str    
 12  production_companies   46478 non-null  str    
 13  production_countries   46478 non-null  str    
 14  release_date           46394 non-null  str    
 15  revenue      

In [305]:
df.insert(0, 'imdb_id', df.pop('imdb_id')) # rearrange the columns for better readability
df.insert(1, 'title',df.pop('title'))
df.insert(2, 'tagline',df.pop('tagline'))


In [306]:
unique_count = df['imdb_id'].nunique()

print(unique_count)  

45415


In [307]:
# Filter for ALL rows where the ID is repeated
all_duplicates = df[df.duplicated(subset=['imdb_id'], keep=False)]

# Sort by 'user_id' so the matching ones are printed side-by-side
print(all_duplicates.sort_values(by='imdb_id').head())

         imdb_id                  title tagline  adult belongs_to_collection  \
36494  tt0009369                 Mickey     NaN  False                   NaN   
36495  tt0009369                 Mickey     NaN  False                   NaN   
36393  tt0011865  Why Change Your Wife?     NaN  False                   NaN   
36392  tt0011865  Why Change Your Wife?     NaN  False                   NaN   
36394  tt0012465         Miss Lulu Bett     NaN  False                   NaN   

       budget                                             genres homepage  \
36494  250000  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...      NaN   
36495  250000  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...      NaN   
36393       0  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...      NaN   
36392       0  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...      NaN   
36394       0  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...      NaN   

           id original_language  ...  \
36494   54242   

In [308]:
df = df.drop_duplicates(subset=['id'], keep='first')

df.info()

<class 'pandas.DataFrame'>
Index: 45432 entries, 0 to 46481
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   imdb_id                45415 non-null  str    
 1   title                  45429 non-null  str    
 2   tagline                20401 non-null  str    
 3   adult                  45432 non-null  str    
 4   belongs_to_collection  4488 non-null   str    
 5   budget                 45432 non-null  str    
 6   genres                 45432 non-null  str    
 7   homepage               7774 non-null   str    
 8   id                     45432 non-null  str    
 9   original_language      45421 non-null  str    
 10  original_title         45432 non-null  str    
 11  overview               44478 non-null  str    
 12  popularity             45429 non-null  object 
 13  poster_path            45046 non-null  str    
 14  production_companies   45429 non-null  str    
 15  production_countri

In [309]:
df = df.drop_duplicates(subset=['imdb_id'], keep='first')

df.info()

<class 'pandas.DataFrame'>
Index: 45416 entries, 0 to 46481
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   imdb_id                45415 non-null  str    
 1   title                  45413 non-null  str    
 2   tagline                20398 non-null  str    
 3   adult                  45416 non-null  str    
 4   belongs_to_collection  4486 non-null   str    
 5   budget                 45416 non-null  str    
 6   genres                 45416 non-null  str    
 7   homepage               7773 non-null   str    
 8   id                     45416 non-null  str    
 9   original_language      45405 non-null  str    
 10  original_title         45416 non-null  str    
 11  overview               44464 non-null  str    
 12  popularity             45413 non-null  object 
 13  poster_path            45035 non-null  str    
 14  production_companies   45413 non-null  str    
 15  production_countri

In [310]:
selected_features = ['title','keywords','genres','original_language']

In [311]:
X = df[selected_features]



Convert language code to full form for better understanding

In [312]:
import pycountry
lang = X['original_language'].unique()
print(lang)

def get_lang(code):
    if not isinstance(code, str) or len(str(code).strip()) == 0:
        return "Unknown Language"
        
    try:
        clean_code = code.strip().lower() 
        language = pycountry.languages.get(alpha_2=clean_code) or pycountry.languages.get(alpha_3=clean_code)
        return language.name if language else "Unknown Language"
    except LookupError:
        return "Unknown Language"
X['original_language'] = X['original_language'].apply(get_lang)

lang = X['original_language'].unique()
print(lang)

<StringArray>
['en', 'fr', 'zh', 'it', 'fa', 'nl', 'de', 'cn', 'ar', 'es', 'ru', 'sv', 'ja',
 'ko', 'sr', 'bn', 'he', 'pt', 'wo', 'ro', 'hu', 'cy', 'vi', 'cs', 'da', 'no',
 'nb', 'pl', 'el', 'sh', 'xx', 'mk', 'bo', 'ca', 'fi', 'th', 'sk', 'bs', 'hi',
 'tr', 'is', 'ps', 'ab', 'eo', 'ka', 'mn', 'bm', 'zu', 'uk', 'af', 'la', 'et',
 'ku', 'fy', 'lv', 'ta', 'sl', 'tl', 'ur', 'rw', 'id', 'bg', 'mr', 'lt', 'kk',
 'ms', 'sq',  nan, 'qu', 'te', 'am', 'jv', 'tg', 'ml', 'hr', 'lo', 'ay', 'kn',
 'eu', 'ne', 'pa', 'ky', 'gl', 'uz', 'sm', 'mt', 'hy', 'iu', 'lb', 'si']
Length: 90, dtype: str
<StringArray>
[               'English',                 'French',                'Chinese',
                'Italian',                'Persian',                  'Dutch',
                 'German',       'Unknown Language',                 'Arabic',
                'Spanish',                'Russian',                'Swedish',
               'Japanese',                 'Korean',                'Serbian',
       

In [313]:
X.head()



,title,keywords,genres,original_language
0,Toy Story,"[{'id': 931, 'name': 'jealousy'}, {'id': 4290,...","[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",English
1,Jumanji,"[{'id': 10090, 'name': 'board game'}, {'id': 1...","[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",English
2,Grumpier Old Men,"[{'id': 1495, 'name': 'fishing'}, {'id': 12392...","[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",English
3,Waiting to Exhale,"[{'id': 818, 'name': 'based on novel'}, {'id':...","[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",English
4,Father of the Bride Part II,"[{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...","[{'id': 35, 'name': 'Comedy'}]",English


In [314]:
def get_names(keywordList):
    if pd.isna(keywordList):
        return []
    if isinstance(keywordList,str):
        try:
            keywordList = ast.literal_eval(keywordList)
        except (ValueError, SyntaxError):
            return [] 
    if isinstance(keywordList, list):
            return [i['name'] for i in keywordList if isinstance(i, dict) and 'name' in i]

    return []



X['keywords'] = X['keywords'].apply(get_names)
X['genres']=X['genres'].apply(get_names)


In [315]:
X.head()

,title,keywords,genres,original_language
0,Toy Story,"[jealousy, toy, boy, friendship, friends, riva...","[Animation, Comedy, Family]",English
1,Jumanji,"[board game, disappearance, based on children'...","[Adventure, Fantasy, Family]",English
2,Grumpier Old Men,"[fishing, best friend, duringcreditsstinger, o...","[Romance, Comedy]",English
3,Waiting to Exhale,"[based on novel, interracial relationship, sin...","[Comedy, Drama, Romance]",English
4,Father of the Bride Part II,"[baby, midlife crisis, confidence, aging, daug...",[Comedy],English


In [316]:
# Replacing the null valuess with null string
for feature in selected_features:
    X[feature] = X[feature].fillna('')
X['keywords_str'] = X['keywords'].apply(lambda x: ' '.join(x))
X['genres_str'] = X['genres'].apply(lambda x: ' '.join(x))

In [317]:
# combining all the 5 selected features
combined_features = X['title'] + ' ' + X['keywords_str'] + ' ' + X['genres_str'] + ' ' + X['original_language'] 
combined_features

0        Toy Story jealousy toy boy friendship friends ...
1        Jumanji board game disappearance based on chil...
2        Grumpier Old Men fishing best friend duringcre...
3        Waiting to Exhale based on novel interracial r...
4        Father of the Bride Part II baby midlife crisi...
                               ...                        
46477              Subdue tragic love Drama Family Persian
46478    Century of Birthing artist play pinoy Drama Ta...
46479              Betrayal  Action Drama Thriller English
46480                           Satan Triumphant   English
46481                                   Queerama   English
Length: 45416, dtype: str

In [318]:
vectorizer = TfidfVectorizer()

feature_vectors = vectorizer.fit_transform(combined_features)

In [ ]:
print(feature_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 479286 stored elements and shape (45416, 28774)>
  Coords	Values
  (0, 25953)	0.7725048404010462
  (0, 24294)	0.13558465329866906
  (0, 13106)	0.16724654310943468
  (0, 3458)	0.3056586290779923
  (0, 9767)	0.14178142228937016
  (0, 9766)	0.15669257983917947
  (0, 21314)	0.1794803973690295
  (0, 17512)	0.1961042287260832
  (0, 7481)	0.1961042287260832
  (0, 17484)	0.12160321158464439
  (0, 5423)	0.20084992687692343
  (0, 25738)	0.11721680446129058
  (0, 14682)	0.12679899488889965
  (0, 1230)	0.10773175598945631
  (0, 5422)	0.058104277577422245
  (0, 8865)	0.09232503538938516
  (0, 8267)	0.03494330029494744
  (1, 17484)	0.17745546892022937
  (1, 8865)	0.13472984993242543
  (1, 8267)	0.05099273003282515
  (1, 13378)	0.41898784101081377
  (1, 3176)	0.30519644316097355
  (1, 10010)	0.22620989223863175
  (1, 7181)	0.28234572707814815
  (1, 2371)	0.16255061530871598
  :	:
  (45410, 21372)	0.6615369680937159
  (45411, 8865)	0.218584

In [ ]:
# getting the similarity scores using cosine similarity
similarity = cosine_similarity(feature_vectors, feature_vectors)

